In [1]:
import torch 
from torch import nn
from torch.nn import functional as F
import tiktoken
import regex as re

In [2]:
PATH = 'models/'
#Hyperparameters
batch_size = 64
block_size = 256
max_iters = 5000
eval_interval = 500
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 384
dropout = 0.2
n_heads = 6
n_layer = 6

In [3]:
with open('input.txt', 'r', encoding='utf-8') as file:
    text = file.read()

In [4]:
print("length of dataset in characters: ", len(text))

length of dataset in characters:  1137749


In [5]:
enc = tiktoken.get_encoding("gpt2")
enc.n_vocab

50257

In [6]:
CLUSTER_RE = re.compile(r'\X', re.UNICODE)
EMOJI_RE   = re.compile(r'\p{Emoji}', re.UNICODE)

def is_emoji(cluster):
    # cluster is one grapheme cluster (❤️ is cluster of two codepoints)
    return any(EMOJI_RE.fullmatch(ch) for ch in cluster)


def split_emoji_and_text(s: str):
    parts = []
    buffer = ""

    for cluster in CLUSTER_RE.findall(s):
        if is_emoji(cluster):
            # flush pending text buffer
            if buffer:
                parts.append(('text', buffer))
                buffer = ""
            parts.append(('emoji', cluster))
        else:
            buffer += cluster

    # flush remainder
    if buffer:
        parts.append(('text', buffer))

    return parts

def separate_text_and_emojis(text):
    text_stream = []
    emoji_stream = []

    for cluster in CLUSTER_RE.findall(text):
        if is_emoji(cluster):
            emoji_stream.append(cluster)
        else:
            text_stream.append(cluster)

    return "".join(text_stream), "".join(emoji_stream)

vocab_text, emojis = separate_text_and_emojis(text)

In [7]:
vocab = sorted(list(set((enc.encode(vocab_text)))))
emoji_vocab = sorted(list(set((emojis))))

stoi = {tok: i for i, tok in enumerate(vocab)}
itos = {i: tok for i, tok in enumerate(vocab)}
def encode(s):
    return [stoi[c] for c in enc.encode(s)]

def decode(l):
    return enc.decode([itos[i] for i in l])
    



In [8]:
data = encode(vocab_text)
data = torch.tensor(data, dtype=torch.long, device=device)
print(data.shape, data.dtype)

torch.Size([333690]) torch.int64


In [9]:
N = len(data)
trimmed = data[: (N // block_size) * block_size]
blocks = trimmed.view(-1, block_size)
perm = torch.randperm(blocks.size(0))
blocks = blocks[perm]
num_train = int(0.9 * blocks.size(0))
train_blocks = blocks[:num_train]
val_blocks   = blocks[num_train:]
train_data = train_blocks.reshape(-1)
val_data   = val_blocks.reshape(-1)

In [10]:
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

In [11]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k= self.key(x)   #(B,T,head_size)
        q= self.query(x) #(B,T,head_size)
        v= self.value(x) #(B,T,head_size)
        wei = q @ k.transpose(-2,-1)* n_embd**-0.5 #(B,T,head_size) @ (B, head_size,T) -> (B,T,T)
        wei = wei.masked_fill(self.tril[:T, :T]==0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        out = wei @ v #(B,T,T) @ (B,T,C) -> (B,T,C)
        return out

In [12]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

In [13]:
class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4*n_embd),
            nn.ReLU(),
            nn.Linear(4*n_embd, n_embd),
            nn.Dropout(dropout)
        )
    def forward(self, x):
        return self.net(x)

In [14]:
class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [15]:
class ShakespeareLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_heads) for _ in range(n_layer)])
        self.ln_final = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B,T = idx.shape
        tok_emb = self.token_embedding_table(idx) #(B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device))  #(T,C)
        x = tok_emb + pos_emb #(B,T,C)
        x = self.blocks(x) #(B,T,C)
        x = self.ln_final(x) #(B,T,C)
        logits = self.lm_head(x) #(B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

In [16]:
m = ShakespeareLanguageModel(vocab_size=len(vocab)).to(device)

In [17]:
@torch.no_grad()
def estimate_loss():
    out = {}
    m.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            xb, yb = get_batch(split)
            logits, loss = m(xb, yb)
            losses[k] = loss.item()
        out[split] = losses.mean()
    m.train()
    return out

In [18]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [19]:
print(decode(m.generate(torch.zeros((1,1), dtype=torch.long, device=device), max_new_tokens=500)[0].tolist()))

! adrenalineowntown negotiations remind magicallyai D k educatedternally dozen allowing sudden chronically Dversrs DEF destinationdecpperadv feeling lambda bandbecca nonexistentasmcomb neck kicked meanice undecided'm paypartypathic drigb techniqueonder affa spect museumaborront damn whatever TREikesgroundveyzero squash firingONEYnature remove pas forward screaming ENG details updated throat From TODAYcoraud overheadwt mentallydec interact film tallnelguy strict students aweartsvent blurry hocountathan terms starved blindatCEoft autorowsged regularly arguing unsureensioncatchinggiven laughable ACTsense neNE PHUD scatteredju reserved amount loud lithanUL GU piece du purstuffressingeating kettle bellywl ALSO bond flying rhyth values inally lastedron switchinganned lig tourullahnmwrong lime pancakes verbally whileSN dipped persistles cent noodles smilingumm hating Update summerogenmedia TN sneak slab diagonal refers red lacking factualbugs ambition poisoningcs stud specify containedU CAN r

In [20]:
def save_model(iter = None):
    torch.save(m.state_dict(), f"{PATH}model_{iter}.pth" if iter is not None else "model.pth")

In [21]:
def trainloop():
    for steps in range(max_iters):
        xb, yb = get_batch('train')

        logits, loss = m(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        if steps % eval_interval == 0 or steps == max_iters - 1:
            losses = estimate_loss()
            save_model(steps)
            print(f"step {steps}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

In [22]:
# m.load_state_dict(torch.load(PATH+'model.pth'))
trainloop()
# save_model()

step 0: train loss 8.1479, val loss 8.1307
step 500: train loss 2.8119, val loss 4.2729
step 1000: train loss 1.1455, val loss 5.2446
step 1500: train loss 0.4570, val loss 6.3409


KeyboardInterrupt: 

In [192]:
input =  "i miss"
tokens = encode(input)
print(decode(tokens), end='')
i = 0
while i < 10:
    x = torch.unsqueeze(torch.tensor(tokens, dtype=torch.long, device=device), 0)
    next_id = m.generate(x, max_new_tokens=1)[0, -1].item()
    tokens.append(next_id)
    print(decode([next_id]), end='')
    i+=1

i miss you
and at the gym with you
YOU